# Main Pipeline Experiment

main.py의 process_item 로직을 단계별로 분리하여 실험할 수 있는 노트북

## Steps
1. **Config & Data Loading** - 설정 및 데이터 로드
2. **SELECT Hint Extraction** - NLQ에서 SELECT fragments 추출
3. **Schema Exploration** - 스키마 탐색 쿼리 실행 (optional)
4. **Fragment Mapping** - fragments → table.column 매핑
5. **Hints Assembly** - evidence, mapping, join keys 조합
6. **Final Prompt Building** - 최종 프롬프트 생성
7. **Model Call** - LLM 호출 및 SQL 추출
8. **Evaluation** - Gold SQL과 비교

In [13]:
import os
import sys
import json
from datetime import datetime
from tqdm import tqdm

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))  # scripts/pipeline/
sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()

# ============================================================
# LOGGING SETUP
# ============================================================
log_buffer = []
LOG_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = os.path.join(os.getcwd(), f"pipeline_log_{LOG_TIMESTAMP}.txt")

def log(msg: str, also_print: bool = False):
    """Add message to log buffer."""
    log_buffer.append(msg)
    if also_print:
        print(msg)

def flush_log():
    """Save log buffer to file."""
    with open(LOG_FILE, 'w', encoding='utf-8') as f:
        f.write("\n".join(log_buffer))
    print(f"Log saved: {LOG_FILE}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Log file: {LOG_FILE}")

Project root: /Users/kyong/Desktop/text-to-sql-analysis
Log file: /Users/kyong/Desktop/text-to-sql-analysis/scripts/pipeline/pipeline_log_20260225_093115.txt


## 1. Configuration

In [14]:

# 처리할 아이템 수 (None = 전체)
LIMIT = 121

# TEST_INDICES = [31, 35, 36, 37]  # 특정 인덱스
# TEST_INDICES = list(range(0, 121))  
TEST_INDICES = list(range(20, 40))  # 인덱스 80~110
USE_SEMANTIC_LAYER = True   

# NLQ 모드: "original" = 기존 question 사용, "verified" = verify_loop_results.json의 final_nlq 사용

# 파이프라인 모드: "full" = 7-step 전체, "simple" = 스키마+힌트 → SQL 직접 생성 (refine 없음)

# Model
MODEL_NAME = "gpt-4o"

# Data paths
DATA_PATH = os.path.join(PROJECT_ROOT, "data/beaver/dw/formatted_data.json")
DATA_DIR = os.path.join(PROJECT_ROOT, "data/beaver/dw")

# MySQL connection (for schema exploration and evaluation)
DB_CONFIG = {
    'host': os.getenv("MYSQL_HOST", "127.0.0.1"),
    'port': int(os.getenv("MYSQL_PORT", 3306)),
    'user': os.getenv("MYSQL_USER", "root"),
    'password': os.getenv("MYSQL_PASSWORD", "")
}

# Output
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs/pipeline_experiment")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Limit: {LIMIT}")


Data path: /Users/kyong/Desktop/text-to-sql-analysis/data/beaver/dw/formatted_data.json
Output dir: /Users/kyong/Desktop/text-to-sql-analysis/outputs/pipeline_experiment
Limit: 121


## 2. Load Data & Components

In [15]:
from openai import OpenAI

from src.utils.semantic_layer import load_semantic_layer
from src.utils.fragment_mapper import map_fragments_to_columns, format_mapping_hint

# Load dataset
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

# Filter dataset
if TEST_INDICES is not None:
    dataset = []
    for i in TEST_INDICES:
        item = full_dataset[i].copy()
        item['original_index'] = i
        dataset.append(item)
elif LIMIT is not None:
    dataset = []
    for i, item in enumerate(full_dataset[:LIMIT]):
        item = item.copy()
        item['original_index'] = i
        dataset.append(item)
else:
    dataset = []
    for i, item in enumerate(full_dataset):
        item = item.copy()
        item['original_index'] = i
        dataset.append(item)

# Load semantic layer
semantic_layer = None
if USE_SEMANTIC_LAYER:
    semantic_layer = load_semantic_layer(DATA_DIR)
    if semantic_layer:
        print(f"Semantic layer loaded")

# OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print(f"Dataset: {len(dataset)} items")
print(f"Sample keys: {list(dataset[0].keys())}")


Semantic layer loaded
Dataset: 20 items
Sample keys: ['question', 'db_id', 'sql', 'oracle_sql', 'gold_tables', 'mapping', 'join_keys', 'formatted_schema', 'original_index']


In [16]:
# ============================================================
# PROMPT TEMPLATES: 7-Step Decomposed Pipeline
# Step1a → Step1b → Step2a → Step2b (+lookup_val) → Step2c → Step3 (Assembly) → Step4
# ============================================================

# ------ Step 1a: NLQ-only Surface Extraction (schema FORBIDDEN) ------
STEP1A_SYSTEM = """You are an information extraction engine for SQL planning.

Hard rules:
- You MUST NOT use any database schema, join hints, or database knowledge.
- Do NOT invent table names or column names.
- Extract only what is explicitly requested or clearly implied by the NLQ.
- Output must be valid JSON, no extra commentary."""

STEP1A_PROMPT = """Task: From the NLQ only, extract surface-level query requirements.

Return JSON with:
- select_fragments: list of OUTPUT ITEMS the user wants to SEE in the result.
- filter_fragments: list of constraints that RESTRICT which rows are returned.
- grouping_dimensions_mentioned: dimensions mentioned with "for each / per / by".
- order_by_fragments: sorting keys and direction.
- distinct_hint: true if NLQ says "unique", "distinct", "no duplicates", "list", etc.

CRITICAL RULES for select_fragments vs grouping_dimensions:
- "for each X" / "per X" / "by X" → goes into grouping_dimensions_mentioned.
- select_fragments should contain the actual values the user wants to see as output columns.
- IMPORTANT: If a field is BOTH an explicit output AND a grouping dimension, include it in BOTH lists.
  Example: "What is the term code, term description... for each term code?"
    → select_fragments: ["term code", "term description", ...]  (term code IS requested as output!)
    → grouping_dimensions: ["term code"]  (term code is ALSO the grouping dimension)
  The same field can appear in both lists when explicitly asked for as output.

Examples:
  NLQ: "List the unique course instructor names, course titles, and the amount of material for each course instructor key and the key of subject offered."
  → select_fragments: ["course instructor names", "course titles", "amount of material"]  (3 items only!)
  → grouping_dimensions: ["course instructor key", "key of subject offered"]
  → "for each course instructor key and the key of subject offered" is NOT a select fragment!

  NLQ: "For each building key, what is the building name and the number of courses?"
  → select_fragments: ["building name", "number of courses"]  (2 items only!)
  → grouping_dimensions: ["building key"]
  → "For each building key" is NOT a select fragment!

  NLQ: "What are the subject titles and total costs for each subject title?"
  → select_fragments: ["subject titles", "total costs"]
  → grouping_dimensions: ["subject title"]
filter_fragments rules:
- Only include REAL filters — conditions that restrict which rows are returned.
- Do NOT include SELECT column names as filters. "new shelf prices" is a column to display, NOT a filter.
- A filter needs an explicit condition: "before 1950", "more than 300", "for department X", etc.

Use these fields for each fragment:
- name: natural-language name (no schema terms)
- intent: scalar|agg|expr|unknown (best guess; can be "unknown")
- source_phrase: the exact phrase (or short span) from the NLQ

NLQ:
<<<{NLQ}>>>"""


# ------ Step 1b: NLQ-only Latent Constraints (schema still FORBIDDEN) ------
STEP1B_SYSTEM = """You generate latent SQL constraints from NLQ without seeing the schema.

Hard rules:
- Do NOT reference any real table/column names.
- Do NOT convert hypotheses into actual WHERE clauses.
- Output must be JSON only.

CRITICAL RULES:
- Do NOT generate latent constraints from SELECT column descriptors or display names.
  "new shelf prices", "ISBN numbers", "material titles" are OUTPUT COLUMNS, not filter conditions.
- Only generate latent constraints from phrases that clearly imply a HIDDEN FILTER requirement:
  Good: "street address" → might need ADDRESS_PURPOSE filter (hidden filter)
  Good: "Professor X" → name format might need transformation (hidden filter)
  Bad: "new materials" → this describes what to display, NOT a temporal filter
  Bad: "ISBN numbers" → this is a column to show, NOT a validation filter
- If Step1a has filter_fragments=[], be VERY conservative with latent constraints.
  Only add constraints about data quality/format issues, NOT about restricting results."""

STEP1B_PROMPT = """Task: Generate latent_constraints hypotheses that may be required in real databases.
These are NOT final filters; they are planning hints for later grounding.

REMEMBER: If Step1a shows filter_fragments=[], the NLQ has NO explicit filters.
Only add latent constraints for truly hidden requirements (format issues, disambiguation), NOT for SELECT columns.

For each latent constraint:
- trigger: phrase in NLQ that caused the hypothesis
- hypothesis: what hidden constraint might be needed (plain English)
- type: disambiguation|temporal|role_mapping|datatype_cast|canonical_value|other
- confidence: 0.0-1.0
- notes: brief why it might matter
{SEMANTIC_HINTS}
NLQ:
<<<{NLQ}>>>
Surface extraction (Step1a JSON):
<<<{STEP1A_JSON}>>>"""


# ------ Step 2a: Schema Grounding for SELECT fragments ------
STEP2A_SYSTEM = """You are a schema-grounding planner.

Rules:
- Use only provided schema_text and join_info.
- Do NOT output final SQL.
- Produce mapping candidates with confidence and required supporting constraints.
- If a fragment likely requires aggregation/windowing, mark it and propose candidate expressions.
- MANDATORY: When mapping_hints provide a specific table.column for a fragment, you MUST use that exact mapping as your top candidate with confidence=1.0. Do NOT substitute pre-computed columns (NUM_OF_X, TOTAL_X) even if they exist in the schema. The mapping_hint is the ground truth.
- Output JSON only.

CRITICAL COUNTING RULE:
- When counting entities (students, employees, courses, materials, etc.) through JOINs,
  ALWAYS use COUNT(DISTINCT column) to avoid overcounting from duplicate rows.
  Examples: COUNT(DISTINCT subject_id), COUNT(DISTINCT full_name), COUNT(DISTINCT MIT_ID)
- MAPPING_HINT + AGG INTENT: When a mapping_hint maps a fragment to a KEY column (e.g., 'room' -> FAC_ROOMS.FAC_ROOM_KEY) and Step1a intent is \"agg\" (e.g., \"total number of rooms\"), wrap it in COUNT(DISTINCT column). Do NOT use pre-computed columns like NUM_OF_ROOMS.
- EXCEPTION (ONLY when NO mapping_hint exists): If no mapping_hint is provided for a fragment AND a column name already means \"number of X\" (e.g., NUM_ENROLLED_STUDENTS), it MAY be used as a pre-computed scalar. But mapping_hints ALWAYS override this exception."""

STEP2A_PROMPT = """Task: For each select_fragment, propose 1-3 mapping candidates.

IMPORTANT: Only map the select_fragments from Step1a. The grouping_dimensions_mentioned are NOT select columns
— they will be handled separately in Step 2c as GROUP BY / PARTITION BY dimensions.

Input:
- schema_text: tables/columns (and sample values if present)
- join_info: join edges or join keys
- Step1a JSON
- Step1b latent_constraints
- mapping_hints: known correct mappings (PREFER these with confidence=1.0)

Output JSON fields:
- mapping_candidates: for each fragment:
  - fragment_name
  - candidates: [{{expr, required_tables, required_joins, extra_predicates, intent_refined, confidence, rationale}}]
  - needs_disambiguation: true/false (and why)
  - needs_window_or_group: none|group|window|unknown
{SEMANTIC_HINTS}
schema_text:
<<<{SCHEMA_TEXT}>>>

join_info:
<<<{JOIN_INFO}>>>

mapping_hints (PREFER these exact mappings with confidence=1.0):
<<<{MAPPING_HINTS}>>>

Step1a:
<<<{STEP1A_JSON}>>>

Step1b:
<<<{STEP1B_JSON}>>>"""





# ------ Step 2a-G: Grain Safety Check (detect SUM/AVG inflation risk) ------
STEP2A_G_SYSTEM = """You are a grain safety analyzer for SQL query planning.

Your job is to detect whether aggregate expressions (SUM, AVG) are at risk of
inflation due to join fan-out (row multiplication from many-to-many or one-to-many joins).

Key concepts:
- COUNT(DISTINCT col) is IMMUNE to join fan-out — it self-corrects for duplicates.
- SUM(col) and AVG(col) are VULNERABLE: if rows are duplicated by JOINs, these aggregates
  count duplicated values multiple times, producing inflated (incorrect) results.
- The "grain" of a table is the natural level of detail (what makes one unique row).
- "Fan-out" occurs when joining table A (1 row per group) to table B (N rows per A row),
  then back to table C — causing A's rows to be multiplied by N.

Rules:
- Analyze ALL aggregates from Step 2a, but only FLAG SUM/AVG expressions.
- COUNT(DISTINCT) is always safe — mark as grain_risk: "none".
- Consider the FULL set of JOINs needed for ALL select fragments combined.
  A SUM may be safe in isolation, but becomes unsafe when the full query joins additional tables.
- If the aggregate's source table rows get multiplied by JOINs needed for OTHER columns, flag it.
- Output JSON only."""

STEP2A_G_PROMPT = """Task: Analyze grain safety for all aggregate expressions from Step 2a.

For each aggregate expression where intent_refined = "agg":

1. Classify the aggregate type:
   - COUNT(DISTINCT ...) → always safe, mark grain_risk: "none"
   - SUM(...) or AVG(...) → requires grain analysis (proceed to step 2)

2. For SUM/AVG aggregates, determine fan-out risk:
   a) measure_table: Which table does the aggregated column belong to?
   b) group_key: What is the GROUP BY key (from Step1a grouping_dimensions)?
   c) all_required_tables: What tables does the FULL query need (across ALL fragments)?
   d) fan_out_analysis: When all required tables are JOINed together, do any JOINs
      cause the measure_table's rows to be MULTIPLIED per group_key?

   How to detect fan-out:
   - If measure_table has N rows per group_key, and another JOIN adds M rows per
     measure_table row, then SUM will be inflated by factor M.
   - Example: SPACE_DETAIL (root, grouped by BUILDING_COMPONENT) has many rooms per component.
     JOIN to SPACE_UNIT (via SPACE_UNIT_KEY) → each room maps to one unit (OK so far).
     JOIN to SPACE_SUPERVISOR_USAGE (via DLC_KEY=DEPT_NAMES) → each unit maps to many supervisors.
     Now each SPACE_DETAIL row is multiplied by the number of supervisors for its unit.
     → SUM(ROOM_SQUARE_FOOTAGE) is inflated!

3. For flagged aggregates, propose a remedy:
   - "window": Compute via window function in a subquery BEFORE fan-out joins.
     Best when: measure_table IS the root table (self-aggregation).
     Pattern: SELECT *, SUM(col) OVER (PARTITION BY group_key) AS pre_agg FROM measure_table
   - "cte": Compute in a CTE at the measure table's native grain, then JOIN result.
     Best when: measure_table is DIFFERENT from root table.
     Pattern: WITH agg AS (SELECT join_key, SUM(col) AS total FROM table GROUP BY join_key)

Output JSON:
{{
  "grain_checks": [
    {{
      "fragment_name": "name from Step2a",
      "agg_expr": "the expression, e.g. SUM(TABLE.COL)",
      "agg_type": "SUM|AVG|COUNT_DISTINCT",
      "measure_table": "TABLE_NAME",
      "measure_column": "COLUMN_NAME",
      "group_key": "TABLE.COLUMN from grouping dimensions",
      "grain_risk": "none|low|medium|high",
      "risk_reason": "explanation of why this is/isn't at risk",
      "remedy": "safe|window|cte",
      "remedy_detail": "specific SQL pattern to use, or N/A if safe"
    }}
  ],
  "has_grain_risks": true|false,
  "pre_aggregation_plan": {{
    "window_subqueries": [
      {{
        "base_table": "TABLE to wrap in subquery",
        "pre_agg_expressions": ["SUM(col) OVER (PARTITION BY key) AS alias"],
        "purpose": "why this pre-aggregation is needed"
      }}
    ],
    "cte_aggregations": [
      {{
        "cte_name": "suggested CTE name",
        "source_table": "TABLE",
        "group_by_key": "COLUMN to group by",
        "agg_expressions": ["SUM(col) AS alias", "COUNT(DISTINCT col) AS alias"],
        "join_back_on": "key to join CTE back to main query"
      }}
    ]
  }},
  "summary": "brief explanation of findings"
}}

Inputs:

schema_text:
<<<{SCHEMA_TEXT}>>>

join_info:
<<<{JOIN_INFO}>>>

Step1a (grouping dimensions = GROUP BY keys):
<<<{STEP1A_JSON}>>>

Step2a (all aggregate mappings — analyze these for grain safety):
<<<{STEP2A_JSON}>>>

mapping_hints:
<<<{MAPPING_HINTS}>>>"""


# ------ Step 2b: Filter Grounding + lookup_val Planning ------
STEP2B_SYSTEM = """You are a filter-grounding and value-resolution planner.

Rules:
- Use provided schema_text only.
- Do NOT output final SQL.
- For each filter fragment, propose candidate (table.column) mappings.
- If literal value is uncertain, propose lookup_val calls or fallback match strategy (LIKE, lower()).
- Output JSON only.

CRITICAL RULES:
- Only ground REAL filters from Step1a's filter_fragments and CONFIRMED latent constraints.
- Do NOT create WHERE filters from grouping dimensions ("for each building key" = GROUP BY, NOT WHERE).
- Do NOT create WHERE filters from SELECT column names ("new shelf prices" = output column, NOT filter).
- Do NOT create WHERE filters for generic category names ("department", "school", "course level") 
  when they appear as grouping dimensions ("for each department" = GROUP BY all departments, NOT filter to one).
- "SIS courses", "TIP subjects", "CIS courses" → the table name IS the filter. Do NOT add WHERE text filters.
- If filter_fragments from Step1a is empty, be VERY conservative — only ground latent constraints
  that represent truly hidden predicates (like ADDRESS_PURPOSE='STREET'), NOT hypothetical ones.
- MANDATORY: When mapping_hints provide a specific table.column for a filter fragment (e.g., 'fall term' -> ACADEMIC_TERMS_ALL.TERM_CODE), you MUST use that table.column as your top candidate. Do NOT substitute with convenience columns (IS_OFFERED_FALL_TERM, IS_CURRENT_TERM, etc.) when a mapping_hint points to a different table/column."""

STEP2B_PROMPT = """Task: Ground each filter_fragment AND CONFIRMED latent_constraints into candidate schema predicates.

IMPORTANT:
- Only ground latent_constraints that represent REAL hidden predicates (e.g., ADDRESS_PURPOSE filtering, name format).
- Do NOT ground latent constraints that are about SELECT columns or grouping dimensions.
- If Step1a filter_fragments is empty, only include latent constraints with confidence >= 0.8 and type = disambiguation|role_mapping|canonical_value.

For lookup_plan, ALWAYS include the most likely table.column to search.
If a filter involves a department name, person name, or any specific entity, include a lookup_plan entry.

Output JSON fields:
- resolved_filters: list where each item has:
  - filter_name
  - column_candidates: [{{table, column, operator, value_strategy, confidence, rationale}}]
  - lookup_plan: if needed, a sequence of lookup_val calls:
    [{{table, column, query_string, match_mode(exact|contains|case_insensitive), fallback}}]
  - final_recommendation: (still not final SQL) best guess mapping + value strategy
{SEMANTIC_HINTS}
schema_text:
<<<{SCHEMA_TEXT}>>>

mapping_hints (MUST use these exact mappings for filter grounding):
<<<{MAPPING_HINTS}>>>

Step1a:
<<<{STEP1A_JSON}>>>

Step1b:
<<<{STEP1B_JSON}>>>"""


# ------ Step 2c: GROUP BY vs WINDOW vs plain SELECT ------
STEP2C_SYSTEM = """You are a logical plan selector for SQL generation.

Rules:
- Decide whether the query should preserve row-level detail or collapse to group-level rows.
- Prefer WINDOW functions when a group-level measure must be attached while keeping row-level columns.
- Produce 2 candidate plans (GROUP and WINDOW) when ambiguous, with a decision score.
- Output JSON only.

===== PRIMARY DECISION RULE =====
DEFAULT = GROUP BY. The vast majority (~85%) of SQL queries with scalar columns + aggregates + grouping dimensions use GROUP BY, NOT WINDOW.

CASE 1 — GROUP BY (DEFAULT — use this unless a rare exception applies):
  When you have scalar columns + aggregate functions + grouping dimensions → use GROUP BY.
  ALL non-aggregated SELECT columns MUST be in GROUP BY (MySQL ONLY_FULL_GROUP_BY requires this).
  The grouping dimension keys ALSO go in GROUP BY (even if not displayed in SELECT output).
  Pattern: SELECT col1, col2, COUNT(DISTINCT x), SUM(y) FROM ... GROUP BY col1, col2, grouping_key
  COUNT(DISTINCT col) works perfectly in GROUP BY.
  
  EXCEPTION — switch to WINDOW (CASE 2) if:
  The scalar SELECT columns are MORE GRANULAR (more detailed) than the grouping dimension.
  Example: SELECT material_title, isbn, price, SUM(price) OVER(PARTITION BY subject_title)
    → Each subject has MANY materials. GROUP BY would include title+isbn+price, making SUM 
      equal to just price (useless). WINDOW preserves per-material detail with per-subject total.
  Rule: If including all scalar columns in GROUP BY would make the aggregate trivial
  (e.g., SUM of one row = the value itself), use WINDOW instead.

CASE 2 — WINDOW (RARE — only when ALL conditions below are met):
  a) The aggregate does NOT need DISTINCT (MySQL does NOT support COUNT(DISTINCT) OVER())
  b) AND one of these specific patterns applies:
     - Filtering on an aggregate result (e.g., "more than 100 employees") → WINDOW + subquery wrapping
       Pattern: SELECT * FROM (SELECT DISTINCT col, AGG() OVER(PARTITION BY key) AS val ...) sub WHERE val > N
     - Pre-computing a per-group total alongside further GROUP BY in outer query
  CRITICAL: COUNT(DISTINCT col) OVER(PARTITION BY ...) is NOT supported in MySQL. NEVER use it.
  If COUNT(DISTINCT) is needed, you MUST use GROUP BY (CASE 1).

CASE 3 — PLAIN (no grouping needed):
  All columns are scalar (no aggregation) → SELECT DISTINCT ...
  OR all columns are aggregates over entire table (no grouping dimension) → SELECT AGG() FROM ...

CASE 4 — SUBQUERY (aggregate at COARSER granularity than detail columns):
  WHEN: The NLQ says "for each X" (grouping dimension = X) and requests an aggregate like
  "number of Y" alongside scalar detail columns (dept_name, school_name, etc.) that are
  MORE GRANULAR than X.
  PROBLEM: If you GROUP BY all scalar columns + X, the COUNT gets split by (X, dept, school, ...)
  instead of being per X only. The NLQ intended the count to be per X.
  SOLUTION: Compute the aggregate in a derived table/subquery grouped by X only,
  then JOIN back to the detail rows.
  Pattern:
    SELECT detail.col1, detail.col2, agg.total_count
    FROM main_table detail
    JOIN (SELECT grouping_key, COUNT(DISTINCT entity) AS total_count
          FROM ... GROUP BY grouping_key) agg
    ON detail.grouping_key = agg.grouping_key
  HOW TO DETECT:
    - Count the grouping_dimensions (N_group) and scalar SELECT columns (N_scalar)
    - If N_scalar > N_group AND the aggregate is "number of" / "total" / "count" with
      "per X" / "for each X" where X = grouping dimension only → CASE 4
    - The scalar columns come from DIFFERENT tables than the grouping dimension
  Example:
    NLQ: "For each term code, list dept name, school name, and the number of subjects"
    → grouping = term_code (1 dim), scalars = dept_name, school_name (more granular)
    → COUNT should be per term_code only → use subquery
    WRONG:  GROUP BY term_code, dept_name, school_name → count split by dept
    RIGHT:  JOIN (SELECT term_code, COUNT(DISTINCT subject_id) GROUP BY term_code) agg ...


CASE 5 — PRE-AGGREGATE (when Step 2a-G flags grain risks):
  WHEN: Step 2a-G grain_checks shows has_grain_risks=true for SUM/AVG aggregates.
  WHY: SUM/AVG through fan-out JOINs produce inflated (incorrect) results.
  COUNT(DISTINCT) is safe, but SUM/AVG are NOT — they count duplicated rows multiple times.

  SOLUTION: The plan type becomes "pre_aggregate".
  - SUM/AVG aggregates with grain_risk=high/medium are computed in CTEs or window subqueries
    at their native grain BEFORE the fan-out joins occur.
  - COUNT(DISTINCT) aggregates remain in the main GROUP BY (they are safe).
  - The main query JOINs pre-aggregated results instead of computing raw SUM/AVG.

  DETECTION: If Step 2a-G output contains has_grain_risks=true → MUST use pre_aggregate plan.
  Follow the pre_aggregation_plan from Step 2a-G for specific CTE/window patterns.

  Pattern (CTE):
    WITH agg_cte AS (
      SELECT join_key, SUM(col) AS pre_agg_total
      FROM measure_table GROUP BY join_key
    )
    SELECT ..., agg_cte.pre_agg_total, COUNT(DISTINCT other_col), ...
    FROM root_table
    JOIN ... ON ...
    JOIN agg_cte ON root_table.key = agg_cte.join_key
    GROUP BY ...

  Pattern (Window subquery):
    SELECT sub.*, COUNT(DISTINCT other_col), ...
    FROM (
      SELECT root_table.*, SUM(col) OVER (PARTITION BY group_key) AS pre_agg_total
      FROM root_table
    ) sub
    JOIN ... ON ...
    GROUP BY ...


===== END DECISION RULE =====


Additional rules:
- "the most X" / "the highest X" (superlative, singular) → needs LIMIT 1 or MAX subquery
- When the query has a filter on an aggregate result (e.g., "more than 100 employees"):
  Option A (PREFERRED): Use GROUP BY + HAVING clause:
    SELECT col, COUNT(DISTINCT x) AS cnt FROM ... GROUP BY col HAVING COUNT(DISTINCT x) > 100
  Option B (if HAVING is not sufficient): Use subquery wrapping:
    SELECT * FROM (SELECT col, COUNT(DISTINCT x) AS cnt FROM ... GROUP BY col) sub WHERE sub.cnt > 100

CRITICAL GROUPING RULES:
- When grouping_dimensions_mentioned references KEY columns (e.g., "for each course instructor key"),
  use the actual KEY column for GROUP BY, NOT the display column.
- When mapping_hints map a grouping dimension to a specific KEY column, USE THAT KEY column.
- If GROUP BY is used, ALL non-aggregated SELECT columns MUST be in GROUP BY.
  This is the #1 most common SQL error. Be explicit: list every non-aggregated SELECT column
  and every grouping key in GROUP BY. Include display/name columns even if they depend on a key.
  Example: SELECT school_code, school_name, dept_name, COUNT(DISTINCT x) 
    → GROUP BY school_code, school_name, dept_name"""

STEP2C_PROMPT = """Task: Choose a logical plan style: plain|group|window|subquery|pre_aggregate.

IMPORTANT: When resolving grouping_dims:
- If mapping_hints map "course instructor key" → table.KEY_COLUMN, use that KEY column
- Do NOT substitute display columns (names/titles) for KEY columns
- grouping_dimensions_mentioned from Step1a are the GROUP BY / PARTITION BY dimensions

IMPORTANT: Count how many select_fragments are scalar vs aggregate:
- If there are scalar + aggregate fragments WITH grouping_dimensions → DEFAULT use GROUP BY
  (GROUP BY includes ALL non-agg SELECT columns + grouping key columns)
- If there are ONLY aggregate fragments WITH grouping_dimensions → use GROUP BY
- If there are ONLY scalar fragments WITHOUT grouping → use plain
- WINDOW is RARE: only for aggregate-result filtering (HAVING not sufficient) or pre-computation
- SUBQUERY is RARE: only when CASE 4 criteria are clearly met (see system prompt)

Inputs:
- Step1a requirements (check grouping_dimensions_mentioned carefully)
- select mapping candidates (Step2a output)
- resolved filters (Step2b output)
- latent constraints (Step1b)

MANDATORY SUBQUERY CHECK (evaluate BEFORE choosing a plan):
Answer these questions to determine if subquery plan is needed:
1. How many grouping dimensions are mentioned? (e.g., "for each term code" = 1 dim)
2. How many NON-GROUPING scalar columns are in SELECT? (columns that are NOT grouping dimensions 
   and NOT 1:1 with a grouping dimension). A column is 1:1 if it's from the same table as a grouping key 
   (e.g., building_name is 1:1 with building_key).
3. Do these NON-GROUPING columns come from DIFFERENT tables than the grouping key?
4. Is there an aggregate (count/total/number of) that should be scoped to the grouping dimension only?

SUBQUERY is recommended ONLY when ALL of these are true:
- N_non_grouping_scalars >= 3 (at least 3 scalar columns that are NOT grouping dimensions)
- These non-grouping scalars come from different tables than the grouping key
- The aggregate should be per the grouping dimension only, NOT per the full combination

KEEP GROUP BY when:
- All scalar columns ARE grouping dimensions or 1:1 with them (e.g., school_name with school_code)
- The NLQ lists multiple entities as output AND wants counts per the combination
- Only 0-1 non-grouping scalar columns exist

Example SUBQUERY: "For each term code, list dept name, school name, attr desc, and number of subjects"
  → 1 grouping dim (term_code), 3 non-grouping scalars from different tables → SUBQUERY
Example GROUP BY: "For each building key, list building name and count of courses"
  → 1 grouping dim, 1 scalar (building_name, 1:1 with building_key) → GROUP BY
Example GROUP BY: "For each school code, department, and course level, list counts"
  → 3 grouping dims, 0 non-grouping scalars → GROUP BY

Output JSON fields:
- output_granularity: row|group|unknown (with reasoning)
- scalar_fragments_count: number of scalar (non-aggregate) select fragments
- aggregate_fragments_count: number of aggregate select fragments
- has_grouping_dimensions: true/false
- decision_logic: e.g. "scalar(2) + agg(1) + grouping = group (default)" with actual counts from above
- candidate_plans: [
   {{
     "plan_type": "plain|group|window|subquery|pre_aggregate",
     "grouping_dims": [...],
     "measures": [...],
     "row_detail_columns": [...],
     "how_distinct_is_handled": "...",
     "needs_limit_or_max": true/false,
     "needs_subquery_wrapping": true/false,
     "subquery_grouping_key": "only for subquery plan: the column(s) to GROUP BY in the derived table",
     "subquery_aggregate_expr": "only for subquery plan: e.g. COUNT(DISTINCT subject_id)",
     "pros": [...],
     "cons": [...],
     "score": 0-100
   }}, ...
 ]
- chosen_plan: plan_type
- plan_notes: any extra predicates implied by disambiguation/latent constraints (still not final SQL)

Mapping hints (for resolving grouping dimensions to KEY columns):
<<<{MAPPING_HINTS}>>>

Step1a:
<<<{STEP1A_JSON}>>>

Step2a(select mapping):
<<<{SELECT_MAPPING_JSON}>>>

Step2b(filters):
<<<{FILTER_GROUNDING_JSON}>>>

Step1b(latent):
<<<{STEP1B_JSON}>>>

Step 2a-G (Grain Safety Analysis — CRITICAL: if has_grain_risks=true, you MUST choose pre_aggregate plan):
<<<{GRAIN_CHECKS_JSON}>>>"""








# ------ Step 3: Final Assembly (consolidate all decisions into structured hint) ------
STEP3_SYSTEM = """You are a SQL plan assembler. Your job is to consolidate all analysis steps into ONE clear, final specification that a SQL writer can follow exactly.

Rules:
- Pick the BEST candidate for each select fragment (highest confidence from Step2a)
- For filters: the final_recommendation in Step2b is AUTHORITATIVE. If a filter's source is 'lookup_val_exact', 'lookup_val_similar', or 'lookup_val_word_match', USE THAT EXACT table.column and value — these were verified against the actual database.
- Use the chosen plan type from Step2c
- List all required tables and joins — follow the join_info edges, do NOT skip intermediate/bridge tables
- List all WHERE predicates — but ONLY real confirmed filters, NOT hypothetical ones
- CRITICAL: Do NOT add WHERE filters for department names, program names, or categories
  when the NLQ says "for each department" / "per department" — that is GROUP BY, NOT WHERE.
  Only add a department WHERE filter when a SPECIFIC department name is mentioned (e.g., "history department")
- When mapping_hints provide a specific table.column for a phrase, USE THAT mapping
- Output a structured text specification, NOT JSON, NOT SQL

CRITICAL RULE about SELECT vs GROUP BY/PARTITION BY:
- select_fragments from Step1a are the OUTPUT columns.
- grouping_dimensions_mentioned from Step1a are the GROUP BY / PARTITION BY columns.
- If a field appears in BOTH select_fragments AND grouping_dimensions, include it in BOTH SELECT and GROUP BY.
  Example: "What is the term code... for each term code?" → term_code in SELECT AND GROUP BY.
- If a field appears ONLY in grouping_dimensions (not in select_fragments), it goes in GROUP BY but NOT SELECT.
  Example: "For each building key, what is the building name?" → building_key in GROUP BY only, building_name in SELECT and GROUP BY.

CRITICAL RULE about GROUP BY (DEFAULT for most queries):
- GROUP BY is the DEFAULT plan for queries with scalar columns + aggregates + grouping dimensions.
- When using GROUP BY: ALL non-aggregated SELECT columns MUST appear in GROUP BY.
  This includes display columns even if functionally dependent on a key.
  Example: GROUP BY SCHOOL_CODE, SCHOOL_NAME, DEPT_CODE (all non-agg SELECT cols + grouping keys).
- The grouping dimension key columns (from Step1a) ALSO go in GROUP BY, even if not in SELECT.
- COUNT(DISTINCT col) is fully supported in GROUP BY.

CRITICAL RULE about WINDOW (RARE — only when Step2c specifically chose "window"):
- Only use WINDOW if Step2c chose "window" for a specific reason (e.g., filtering on aggregate result).
- The pattern is: SELECT DISTINCT scalar_col1, scalar_col2, AGG() OVER(PARTITION BY grouping_dim)
- Do NOT use COUNT(DISTINCT col) inside OVER() — MySQL does not support it.

CRITICAL RULE about SUBQUERY (when Step2c chose "subquery"):
- When Step2c chose "subquery", the aggregate must be computed at a COARSER granularity than the detail rows.
- The aggregate (COUNT, SUM, etc.) goes in a derived table/subquery grouped ONLY by the grouping dimension.
- The detail rows are joined to the subquery result.
- Pattern:
    SELECT detail.col1, detail.col2, agg.total_count
    FROM main_table detail
    JOIN other_tables ON ...
    JOIN (SELECT grouping_key, COUNT(DISTINCT entity) AS total_count
          FROM source_table [JOIN ...] [WHERE ...]
          GROUP BY grouping_key) agg
    ON detail.grouping_key = agg.grouping_key
    [WHERE ...]
- The subquery must include the same WHERE filters and JOINs relevant to the aggregate.
- Do NOT put the aggregate in the main SELECT with GROUP BY — that would split the count by detail columns.

CRITICAL SQL RULES:
- If using GROUP BY: ALL non-aggregated SELECT columns MUST appear in GROUP BY
- If counting distinct items with GROUP BY: use COUNT(DISTINCT column)
- CRITICAL MYSQL LIMITATION: COUNT(DISTINCT col) OVER(PARTITION BY ...) is NOT supported in MySQL.
  When using WINDOW plan with COUNT, use COUNT(col) OVER(...) WITHOUT DISTINCT.
  The SELECT DISTINCT at the outer level will handle deduplication.
- When filtering on an aggregate result (e.g., "more than 100 employees"), prefer GROUP BY + HAVING:
  SELECT col, COUNT(DISTINCT x) FROM ... GROUP BY col HAVING COUNT(DISTINCT x) > 100
- Always use = operator with exact values from lookup_val (not LIKE)
- If Step1a distinct_hint=true, set DISTINCT: yes
- Window functions CANNOT appear in WHERE clauses. If filtering on a window result is needed,
  wrap the query: SELECT * FROM (SELECT DISTINCT ..., window_fn AS alias ...) sub WHERE sub.alias > N
- Do NOT include WHERE conditions with placeholder values like {var_name} or "lookup" strategy
  that has no resolved value. Only include WHERE conditions with concrete values.

CRITICAL JOIN RULES:
- Follow join_info edges exactly. If a join is needed but NOT in join_info, check semantic hints.
- FCLT_BUILDING joins to FCLT_ROOMS via FCLT_BUILDING_KEY (NEVER via BUILDING_NAME = BUILDING_ROOM)
- FCLT_BUILDING_HIST joins to FCLT_ROOMS via FCLT_BUILDING_KEY
- When using FCLT_BUILDING_HIST, deduplicate first (multiple rows per building per fiscal period)

CRITICAL RULE about PRE-AGGREGATION (when Step2c chose "pre_aggregate" or Step 2a-G flags grain risks):
- SUM/AVG aggregates flagged by Step 2a-G MUST be pre-computed before fan-out JOINs.
- Follow the pre_aggregation_plan from Step 2a-G exactly.
- For WINDOW remedy: Wrap the base table in a subquery with the window function.
  The outer query references the pre-computed alias instead of raw SUM().
  Pattern:
    FROM (SELECT *, SUM(col) OVER (PARTITION BY group_key) AS total_col FROM base_table) AS sub
    JOIN other_tables ON ...
    GROUP BY sub.group_key, sub.total_col, ...
  IMPORTANT: Include the pre-aggregated alias in GROUP BY (since it's a non-aggregated column).
- For CTE remedy: Create a WITH clause CTE that aggregates at the native grain.
  Pattern:
    WITH cte_name AS (
      SELECT group_key, SUM(col) AS total_col
      FROM source_table
      GROUP BY group_key
    )
    SELECT ..., cte_name.total_col, ...
    FROM root_table
    JOIN cte_name ON root_table.key = cte_name.group_key
    JOIN other_tables ON ...
    GROUP BY ...
  IMPORTANT: cte_name.total_col goes directly in SELECT (no SUM wrapper), since it is already aggregated.
- COUNT(DISTINCT col) expressions remain in the main SELECT with GROUP BY as usual.

- MIT_STUDENT_DIRECTORY joins to FCLT_ROOMS via OFFICE_LOCATION = BUILDING_ROOM (NOT via BUILDING_NAME)"""

STEP3_PROMPT = """Assemble the final SQL specification from all pipeline steps.

Output format (plain text, not JSON):

=== FINAL SQL SPECIFICATION ===
Question: {{original question}}
Plan Type: {{plain/group/window/subquery}}

SELECT columns (from Step1a select_fragments ONLY — do NOT include grouping dimensions as SELECT columns):
1. {{table.column or expression}} -- {{description}}
2. ...

FROM + JOIN (use join_info edges, do NOT skip bridge tables):
- {{table1}}
- JOIN {{table2}} ON {{condition}}
- ...

WHERE conditions (ONLY confirmed filters with concrete values):
- {{table.column operator value}} -- {{reason}}
- ...

GROUP BY / PARTITION BY (from Step1a grouping_dimensions_mentioned, mapped to KEY columns):
- If plan=group (DEFAULT): GROUP BY {{ALL non-aggregated SELECT columns}} + {{grouping key columns}}
  CRITICAL: Include EVERY non-aggregated column from SELECT in GROUP BY, even display names.
- If plan=window (RARE): PARTITION BY {{key_columns}} (inside the AGG() OVER() expression)
- If plan=subquery: The aggregate goes in a derived table subquery grouped ONLY by the grouping dimension key.
  The main query JOINs to this subquery. Do NOT compute the aggregate directly in the main SELECT — instead,
  reference the subquery alias column (e.g., agg.total_subjects) in the main SELECT columns list.
  IMPORTANT: You MUST include the subquery alias column in the SELECT columns above (e.g., "5. agg.total_subjects -- number of subjects").
  Specify the subquery separately:
  SUBQUERY: SELECT grouping_key, AGG(col) AS alias FROM ... [WHERE ...] GROUP BY grouping_key
- If plan=plain: N/A

ORDER BY:
- {{columns + direction or "N/A"}}


PRE-AGGREGATION (if Step 2a-G flagged grain risks):
- List each CTE or window subquery needed
- For each: source table, GROUP BY key, aggregate expression, alias
- Specify how the pre-aggregated result joins back to the main query

DISTINCT: {{yes/no}}
LIMIT: {{value or "N/A"}}
SUBQUERY WRAPPING: {{yes/no — if filtering on aggregate/window result}}

Notes:
- {{any special instructions}}
=== END ===

Inputs:

Question: {QUESTION}

Step1a (surface extraction — check distinct_hint and grouping_dimensions_mentioned):
<<<{STEP1A_JSON}>>>

Step2a (select grounding — these are the OUTPUT columns only):
<<<{STEP2A_JSON}>>>

Step2b (filter grounding — final_recommendation is AUTHORITATIVE, especially if source=lookup_val_*):
<<<{STEP2B_JSON}>>>

Step2c (plan decision — FOLLOW the chosen_plan type exactly):
<<<{STEP2C_JSON}>>>

Semantic layer hints:
{SEMANTIC_HINTS}

Mapping hints (USE these exact mappings when available — especially for KEY column grouping):
{MAPPING_HINTS}

Join info (follow these edges for JOIN conditions):
{JOIN_INFO}

Step 2a-G (Grain Safety — follow pre_aggregation_plan if has_grain_risks=true):
<<<{GRAIN_CHECKS_JSON}>>>"""


# ------ Step 4: Final SQL Rendering (receives assembled spec) ------
STEP4_SYSTEM = """You are an expert MySQL SQL writer.

Rules:
- Follow the SQL specification EXACTLY as given — do not add or remove columns, conditions, or joins.
- Use only the tables, columns, joins, and conditions specified.
- Produce exactly one SQL query and nothing else.
- Do not include explanations.
- If the spec says DISTINCT, use SELECT DISTINCT.
- If the spec says WINDOW/PARTITION BY, use window functions with AGG() OVER(PARTITION BY ...).
- If the spec says LIMIT, add LIMIT clause.

CRITICAL MySQL rules:
- If GROUP BY is used, ALL non-aggregated SELECT columns MUST be in GROUP BY.
  This is the MOST COMMON error. Double-check: every SELECT column that is NOT inside
  an aggregate function (COUNT, SUM, MIN, MAX, AVG) MUST appear in the GROUP BY clause.
  Include display/name columns too, even if they seem dependent on a key column.
- Use COUNT(DISTINCT col) when the spec says COUNT(DISTINCT) — BUT ONLY in GROUP BY queries.
- NEVER use COUNT(DISTINCT col) inside OVER(PARTITION BY ...) — MySQL does not support this.
  Use COUNT(col) OVER(...) instead. The SELECT DISTINCT handles dedup.
- For date parsing: use STR_TO_DATE(col, '%m/%d/%Y') and EXTRACT(YEAR FROM ...).
- Do NOT use TO_NUMBER, SUBSTR for dates — use MySQL functions.
- Do NOT include placeholder variables like {var_name} — if a WHERE condition has no concrete value, OMIT that condition entirely.
- Do NOT add extra WHERE conditions that were not in the specification.
- Window functions CANNOT appear in WHERE clauses. If the spec says SUBQUERY WRAPPING: yes,
  wrap the inner query and filter in the outer WHERE.
- JOIN conditions must match the specification exactly — never join BUILDING_NAME = BUILDING_ROOM.
- FCLT_BUILDING always joins to FCLT_ROOMS via FCLT_BUILDING_KEY = FCLT_BUILDING_KEY.
- Do NOT add columns to SELECT that are not in the specification.
- If the spec includes PRE-AGGREGATION CTEs or window subqueries, implement them exactly:
  - CTE pattern: WITH cte_name AS (SELECT key, AGG(col) FROM table GROUP BY key) SELECT ...
  - Window subquery pattern: FROM (SELECT *, AGG(col) OVER (PARTITION BY key) AS alias FROM table) sub
  - Pre-aggregated columns from CTEs/subqueries go directly in SELECT (no extra SUM wrapper).
  - Include pre-aggregated alias columns in GROUP BY when they are non-aggregated in the outer query."""

STEP4_PROMPT = """Generate the final MySQL SQL query following this specification exactly.

{ASSEMBLED_SPEC}

Schema for reference (column names must match exactly):
<<<{SCHEMA_TEXT}>>>

Return SQL only. No explanation."""




# ------ Step 5: Refine (execute SQL → analyze error → LLM fix → retry) ------
REFINE_SYSTEM = """You are an expert MySQL SQL debugger and fixer.

Your task is to fix a SQL query that failed to execute or returned empty results.

Rules:
- Fix ONLY the specific error mentioned in the error analysis.
- Do NOT change the query structure unnecessarily.
- Keep all original columns, tables, and JOIN conditions unless they are the cause of the error.
- If the error is about GROUP BY, ensure ALL non-aggregated SELECT columns are in GROUP BY.
- If the error is about unknown column/table, check the schema carefully.
- If the result was empty (0 rows), check WHERE conditions and JOIN conditions.
- Produce exactly one corrected SQL query and nothing else.
- Do not include explanations."""

REFINE_PROMPT = """The following SQL query had an issue. Please fix it based on the error analysis.

Original Question: {QUESTION}

SQL Query:
```sql
{SQL}
```

Error Analysis:
{ERROR_ANALYSIS}

Schema for reference:
<<<{SCHEMA_TEXT}>>>

Join info:
{JOIN_INFO}


Return only the corrected SQL query. No explanations."""

# Step names for logging
STEP_NAMES = {
    "step1a": "Step 1a: Surface Extraction (NLQ-only)",
    "step1b": "Step 1b: Latent Constraints (NLQ-only)",
    "step2a": "Step 2a: Schema Grounding (SELECT)",
    "step2a_g": "Step 2a-G: Grain Safety Check",
    "step2b": "Step 2b: Filter Grounding + lookup_val",
    "step2b_updated": "Step 2b (Updated): Filter Grounding after lookup_val",
    "step2c": "Step 2c: GROUP BY vs WINDOW vs SUBQUERY Decision",
    "step3": "Step 3: Final Assembly",
    "step4": "Step 4: Final SQL Rendering",
    "refine": "Step 5: Refine (execute → fix → retry)",
    "refine_attempt_1": "Step 5: Refine Attempt 1",
    "refine_attempt_2": "Step 5: Refine Attempt 2",
}

print(f"Loaded {len(STEP_NAMES)} step prompt templates (including refine)")

Loaded 12 step prompt templates (including refine)


In [17]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================
from src.agent.column_value_lookup import lookup_column_values, format_lookup_result


def call_llm(system_msg, user_msg, step_name="", model="gpt-4o", json_mode=True, max_tokens=2000):
    """
    Unified LLM call with logging.
    Returns parsed dict (json_mode=True) or raw string (json_mode=False).
    """
    step_label = STEP_NAMES.get(step_name, step_name)
    log(f"\n[{step_label}]")
    log(f"  [MODEL] {model}")
    log(f"  [SYSTEM PROMPT]")
    log(system_msg)
    log(f"  [USER PROMPT] ({len(user_msg)} chars)")
    log(user_msg)

    kwargs = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        "temperature": 0,
        "max_tokens": max_tokens,
    }
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    try:
        response = client.chat.completions.create(**kwargs)
        content = response.choices[0].message.content.strip()

        if json_mode:
            result = json.loads(content)
            log(f"\n  [LLM RESPONSE]")
            log(json.dumps(result, indent=2, ensure_ascii=False))
            return result
        else:
            log(f"\n  [LLM RESPONSE]")
            log(content)
            return content

    except Exception as e:
        error_msg = str(e)[:300]
        log(f"  [ERROR] {error_msg}")
        if json_mode:
            return {"error": error_msg}
        else:
            return f"-- Error: {error_msg}"


def run_lookup_val(step2b_result, db_config, db_id="dw"):
    """
    Execute lookup_val calls from Step2b's lookup_plan.
    Returns dict of {filter_name: [lookup_results]} or empty dict.
    """
    if not isinstance(step2b_result, dict):
        return {}

    resolved_filters = step2b_result.get('resolved_filters', [])
    if not resolved_filters:
        return {}

    all_results = {}

    for rf in resolved_filters:
        filter_name = rf.get('filter_name', 'unknown')
        lookup_plan = rf.get('lookup_plan', [])

        if not lookup_plan:
            continue

        filter_results = []
        for plan in lookup_plan:
            table = plan.get('table', '')
            column = plan.get('column', '')
            query_string = plan.get('query_string', '')

            if not table or not column:
                continue

            log(f"\n  [LOOKUP_VAL] {table}.{column} search='{query_string}'")

            try:
                result = lookup_column_values(
                    table=table,
                    column=column,
                    conn_info=db_config,
                    db_id=db_id,
                    search_term=query_string if query_string else None,
                    limit=20
                )

                formatted = format_lookup_result(result)
                log(f"  [LOOKUP RESULT]\n{formatted}")

                filter_results.append({
                    "table": table,
                    "column": column,
                    "query_string": query_string,
                    "result": result
                })
            except Exception as e:
                log(f"  [LOOKUP ERROR] {str(e)[:200]}")
                filter_results.append({
                    "table": table,
                    "column": column,
                    "query_string": query_string,
                    "error": str(e)[:200]
                })

        if filter_results:
            all_results[filter_name] = filter_results

    return all_results


def apply_lookup_results(step2b_result, lookup_results):
    """
    Programmatically apply lookup_val results to step2b filter recommendations.
    This replaces the LLM-based update which was unreliable.
    """
    import copy
    
    if not lookup_results or not isinstance(step2b_result, dict):
        return step2b_result, []
    
    updated = copy.deepcopy(step2b_result)
    changes_made = []
    
    for rf in updated.get('resolved_filters', []):
        filter_name = rf.get('filter_name', '')
        if filter_name not in lookup_results:
            continue
        
        results_for_filter = lookup_results[filter_name]
        
        for lr in results_for_filter:
            result = lr.get('result', {})
            if not result.get('success'):
                continue
            
            table = lr['table']
            column = lr['column']
            query_string = lr.get('query_string', '')
            
            if result.get('exact_match'):
                old_rec = rf.get('final_recommendation', {})
                rf['final_recommendation'] = {
                    'table': table, 'column': column, 'operator': '=',
                    'value': query_string,
                    'value_strategy': f"EXACT match found via lookup_val",
                    'source': 'lookup_val_exact',
                    'original_recommendation': old_rec
                }
                changes_made.append(f"[EXACT] {filter_name}: {table}.{column} = '{query_string}'")
                break
            
            similar = result.get('similar_values', [])
            if similar:
                best_value = similar[0]['value']
                old_rec = rf.get('final_recommendation', {})
                rf['final_recommendation'] = {
                    'table': table, 'column': column, 'operator': '=',
                    'value': best_value,
                    'value_strategy': f"closest SIMILAR value from DB: '{best_value}'",
                    'source': 'lookup_val_similar',
                    'original_recommendation': old_rec
                }
                changes_made.append(f"[SIMILAR] {filter_name}: {table}.{column} = '{best_value}'")
                break
            
            word_matches = result.get('word_matches', {})
            if word_matches:
                if 'ALL_WORDS' in word_matches and word_matches['ALL_WORDS']:
                    best_value = word_matches['ALL_WORDS'][0]['value']
                    match_type = 'ALL_WORDS'
                else:
                    first_key = list(word_matches.keys())[0]
                    best_value = word_matches[first_key][0]['value']
                    match_type = f"word '{first_key}'"
                
                old_rec = rf.get('final_recommendation', {})
                rf['final_recommendation'] = {
                    'table': table, 'column': column, 'operator': '=',
                    'value': best_value,
                    'value_strategy': f"WORD match ({match_type}) from DB: '{best_value}'",
                    'source': 'lookup_val_word_match',
                    'original_recommendation': old_rec
                }
                changes_made.append(f"[WORD_MATCH] {filter_name}: {table}.{column} = '{best_value}'")
                break
    
    return updated, changes_made



def format_join_info(item):
    """Format join_keys from item into readable join info string."""
    join_keys = item.get('join_keys', [])
    if not join_keys:
        return "(no join info)"

    join_lines = []
    for pair in join_keys:
        if isinstance(pair, (list, tuple)) and len(pair) == 2:
            join_lines.append(f"{pair[0]} = {pair[1]}")
    return "\n".join(join_lines) if join_lines else "(no join info)"


def get_semantic_hints(question):
    """Get semantic layer hints for a question, formatted for prompt injection."""
    if not semantic_layer:
        return ""

    rules = semantic_layer.get_matching_rules(question)
    hints = []

    for h in rules.get("hints_for_extraction", []):
        hints.append(f"  - {h}")
    for h in rules.get("hints_for_main", []):
        hints.append(f"  - {h}")

    if not hints:
        return ""

    return "\n[DOMAIN-SPECIFIC RULES - MUST FOLLOW]\n" + "\n".join(hints) + "\n"


def format_mapping_hints(item):
    """Format mapping hints from item."""
    mapping = item.get('mapping', {})
    if not mapping:
        return "(no mapping hints)"

    lines = []
    for phrase, columns in mapping.items():
        cols_str = ', '.join(columns) if isinstance(columns, list) else columns
        lines.append(f"- '{phrase}' → {cols_str}")
    return "\n".join(lines)


def clean_sql(sql):
    """Post-process SQL to fix common issues."""
    import re
    # Remove any remaining {placeholder} patterns (from unresolved template vars)
    cleaned = re.sub(r"AND\s+[A-Za-z_.`]+\s*(=|LIKE|>|<|>=|<=)\s*\{[^}]+\}", "", sql)
    cleaned = re.sub(r"WHERE\s+[A-Za-z_.`]+\s*(=|LIKE|>|<|>=|<=)\s*\{[^}]+\}", "WHERE 1=1", cleaned)
    # Clean up WHERE 1=1 in any position (before GROUP BY, ORDER BY, LIMIT, ;, or end)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(r"WHERE\s+1=1\s+AND\s+", "WHERE ", cleaned)
    cleaned = re.sub(r"WHERE\s+1=1\s*(GROUP|ORDER|LIMIT|HAVING|;|$)", r"\1", cleaned)
    cleaned = re.sub(r"WHERE\s+1=1\s*$", "", cleaned)
    return cleaned.strip()





def execute_sql_for_refine(db_id: str, sql: str, timeout_ms: int = 30000) -> dict:
    """
    Execute SQL against MySQL and return result with error details.
    Used by the refine step to check if SQL runs correctly.
    """
    import mysql.connector

    result = {
        "success": False,
        "row_count": 0,
        "error": None,
        "error_type": None,
        "results": []
    }

    try:
        conn = mysql.connector.connect(
            host=DB_CONFIG['host'],
            port=DB_CONFIG['port'],
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password'],
            database=db_id
        )
        cursor = conn.cursor(dictionary=True)
        cursor.execute(f"SET SESSION MAX_EXECUTION_TIME = {timeout_ms}")
        cursor.execute(sql)
        rows = cursor.fetchall()

        result["success"] = True
        result["row_count"] = len(rows)
        result["results"] = rows[:5]

        if len(rows) == 0:
            result["error_type"] = "empty_result"

        cursor.close()
        conn.close()

    except mysql.connector.Error as e:
        error_msg = str(e)
        result["error"] = error_msg
        if "max_execution_time" in error_msg.lower() or "interrupted" in error_msg.lower():
            result["error_type"] = "timeout"
        else:
            result["error_type"] = "syntax_error"

    except Exception as e:
        result["error"] = str(e)
        result["error_type"] = "syntax_error"

    return result


def refine_sql(sql: str, item: dict, max_retries: int = 2) -> tuple:
    """
    Refine SQL through execution → error analysis → LLM fix loop.

    Process:
    1. Execute SQL against MySQL
    2. If success (rows > 0): done
    3. If syntax error: analyze error → feed to LLM → get fixed SQL → retry
    4. If empty result: analyze WHERE/JOIN → feed to LLM → get fixed SQL → retry
    5. Repeat up to max_retries times

    Args:
        sql: Initial predicted SQL
        item: Dataset item (for schema, question, etc.)
        max_retries: Maximum number of fix attempts (default: 2)

    Returns:
        (final_sql, refine_log_list)
    """
    from src.refine_agent.syntax_fixer import analyze_sql_error, format_syntax_fix_advice
    from src.refine_agent.empty_result_handler import analyze_empty_result, format_empty_result_advice

    db_id = item['db_id']
    question = item['question']
    schema = item.get('formatted_schema', '')
    join_info = format_join_info(item)

    refine_log = []
    current_sql = sql

    for attempt in range(max_retries + 1):  # 0 = initial check, 1..max_retries = fix attempts
        # Execute SQL
        exec_result = execute_sql_for_refine(db_id, current_sql)

        log(f"\n  [REFINE attempt {attempt}] success={exec_result['success']}, "
            f"rows={exec_result['row_count']}, error_type={exec_result.get('error_type')}")

        refine_entry = {
            "attempt": attempt,
            "sql": current_sql,
            "success": exec_result["success"],
            "row_count": exec_result["row_count"],
            "error_type": exec_result.get("error_type"),
            "error": (exec_result.get("error") or "")[:300]
        }

        # Success with rows → done
        if exec_result["success"] and exec_result["row_count"] > 0:
            refine_entry["action"] = "success"
            refine_log.append(refine_entry)
            log(f"  [REFINE] SQL executed successfully with {exec_result['row_count']} rows")
            break

        # No more retries left
        if attempt >= max_retries:
            refine_entry["action"] = "max_retries_reached"
            refine_log.append(refine_entry)
            log(f"  [REFINE] Max retries ({max_retries}) reached, keeping current SQL")
            break

        # Generate error analysis based on error type
        error_analysis = ""
        if exec_result.get("error_type") == "syntax_error":
            analysis = analyze_sql_error(current_sql, exec_result.get("error", ""))
            error_analysis = format_syntax_fix_advice(analysis)
            refine_entry["analysis_type"] = "syntax_error"
        elif exec_result.get("error_type") == "empty_result":
            analysis = analyze_empty_result(current_sql, DB_CONFIG, db_id, question)
            error_analysis = format_empty_result_advice(analysis)
            refine_entry["analysis_type"] = "empty_result"
        elif exec_result.get("error_type") == "timeout":
            error_analysis = ("Query execution timed out. Consider:\n"
                            "1. Simplify JOIN conditions\n"
                            "2. Add more specific WHERE conditions\n"
                            "3. Remove unnecessary subqueries\n"
                            "4. Add LIMIT clause")
            refine_entry["analysis_type"] = "timeout"
        else:
            refine_entry["action"] = "unknown_error_type"
            refine_log.append(refine_entry)
            break

        log(f"  [REFINE] Error analysis:\n{error_analysis}")
        refine_entry["error_analysis"] = error_analysis
        refine_log.append(refine_entry)

        # Call LLM to fix the SQL
        fixed_response = call_llm(
            REFINE_SYSTEM,
            REFINE_PROMPT.format(
                QUESTION=question,
                SQL=current_sql,
                ERROR_ANALYSIS=error_analysis,
                SCHEMA_TEXT=schema,
                JOIN_INFO=join_info
            ),
            step_name=f"refine_attempt_{attempt + 1}",
            json_mode=False,
            max_tokens=3000
        )

        # Extract SQL from response
        fixed_sql = fixed_response.strip()
        if fixed_sql.startswith("```sql"):
            fixed_sql = fixed_sql[6:]
        elif fixed_sql.startswith("```"):
            fixed_sql = fixed_sql[3:]
        if fixed_sql.endswith("```"):
            fixed_sql = fixed_sql[:-3]
        fixed_sql = clean_sql(fixed_sql.strip())

        if fixed_sql and fixed_sql != current_sql:
            log(f"  [REFINE] SQL updated")
            log(f"    OLD: {current_sql[:150]}...")
            log(f"    NEW: {fixed_sql[:150]}...")
            current_sql = fixed_sql
        else:
            log(f"  [REFINE] No change from LLM fix, stopping refine loop")
            break

    return current_sql, refine_log








def validate_grain_safety(predicted_sql: str, grain_checks: dict, item: dict) -> dict:
    """
    Structural validation: check if SUM/AVG results might be inflated by join fan-out.

    Strategy: If grain_checks flagged risks, run a simplified sanity check:
    1. Execute the predicted SQL and get result
    2. For each flagged SUM/AVG, run a "grain-safe" version (pre-aggregated)
    3. Compare the results — if they differ significantly, the original has inflation

    Returns dict with validation results.
    """
    if not grain_checks or not grain_checks.get('has_grain_risks'):
        return {"validated": True, "reason": "no_grain_risks_flagged", "checks": []}

    checks = grain_checks.get('grain_checks', [])
    flagged = [c for c in checks if c.get('grain_risk') in ('high', 'medium')]

    if not flagged:
        return {"validated": True, "reason": "no_high_medium_risks", "checks": []}

    log(f"  [GRAIN VALIDATION] {len(flagged)} aggregate(s) flagged for structural check")

    validation_results = []
    for check in flagged:
        validation_results.append({
            "fragment": check.get('fragment_name', ''),
            "agg_expr": check.get('agg_expr', ''),
            "grain_risk": check.get('grain_risk', ''),
            "remedy": check.get('remedy', ''),
            "remedy_detail": check.get('remedy_detail', '')
        })
        log(f"    → {check.get('fragment_name')}: {check.get('grain_risk')} risk, remedy={check.get('remedy')}")

    return {
        "validated": False,
        "reason": "grain_risks_detected",
        "flagged_count": len(flagged),
        "checks": validation_results
    }


print("Helper functions loaded: call_llm, run_lookup_val, apply_lookup_results, format_join_info, get_semantic_hints, format_mapping_hints, clean_sql, execute_sql_for_refine, refine_sql, validate_grain_safety")

Helper functions loaded: call_llm, run_lookup_val, apply_lookup_results, format_join_info, get_semantic_hints, format_mapping_hints, clean_sql, execute_sql_for_refine, refine_sql, validate_grain_safety


## 4. Run Pipeline (6-Step Decomposed)

In [18]:
def process_item(item: dict, verbose: bool = False) -> dict:
    """
    Process a single item through the decomposed pipeline with grain safety check.
    Step1a → Step1b → Step2a → Step2a-G (Grain Check) → Step2b (+lookup_val) → Step2c → Step3 → Step4 → Step5(refine)
    """
    question = item['question']
    schema = item.get('formatted_schema', '')
    join_info = format_join_info(item)
    mapping_hints = format_mapping_hints(item)
    semantic_hints = get_semantic_hints(question)
    original_index = item.get('original_index', -1)
    gold_query = item.get('sql', item.get('SQL', item.get('query', '')))
    db_id = item['db_id']

    log(f"\n{'='*70}")
    log(f"[ITEM {original_index}] {question}")
    log(f"{'='*70}")
    if semantic_hints:
        log(f"  [SEMANTIC HINTS ACTIVE]\n{semantic_hints}")
    if mapping_hints and mapping_hints != "(no mapping hints)":
        log(f"  [MAPPING HINTS]\n{mapping_hints}")

    steps = {}

    # ===== Step 1a: NLQ Surface Extraction (NO schema) =====
    step1a = call_llm(
        STEP1A_SYSTEM,
        STEP1A_PROMPT.format(NLQ=question),
        step_name="step1a",
        model="gpt-4o",
    )
    steps['step1a'] = step1a

    # ===== Step 1b: Latent Constraints (NO schema, but with semantic hints) =====
    step1b = call_llm(
        STEP1B_SYSTEM,
        STEP1B_PROMPT.format(
            NLQ=question,
            STEP1A_JSON=json.dumps(step1a, ensure_ascii=False),
            SEMANTIC_HINTS=semantic_hints
        ),
        step_name="step1b",
        model="gpt-4o",
    )
    steps['step1b'] = step1b

    # ===== Step 2a: Schema Grounding for SELECT (now includes mapping_hints) =====
    step2a = call_llm(
        STEP2A_SYSTEM,
        STEP2A_PROMPT.format(
            SCHEMA_TEXT=schema,
            JOIN_INFO=join_info,
            MAPPING_HINTS=mapping_hints,
            STEP1A_JSON=json.dumps(step1a, ensure_ascii=False),
            STEP1B_JSON=json.dumps(step1b, ensure_ascii=False),
            SEMANTIC_HINTS=semantic_hints
        ),
        step_name="step2a",
        model="gpt-4o",
    )
    steps['step2a'] = step2a

    
    # ===== Step 2a-G: Grain Safety Check =====
    step2a_g = call_llm(
        STEP2A_G_SYSTEM,
        STEP2A_G_PROMPT.format(
            SCHEMA_TEXT=schema,
            JOIN_INFO=join_info,
            STEP1A_JSON=json.dumps(step1a, ensure_ascii=False),
            STEP2A_JSON=json.dumps(step2a, ensure_ascii=False),
            MAPPING_HINTS=mapping_hints
        ),
        step_name="step2a_g",
        model="gpt-4o",
    )
    steps['step2a_g'] = step2a_g
    has_grain_risks = step2a_g.get('has_grain_risks', False) if isinstance(step2a_g, dict) else False
    if has_grain_risks:
        log(f"  [GRAIN CHECK] ⚠️ Grain risks detected — pre-aggregation will be applied")
        for gc in step2a_g.get('grain_checks', []):
            if gc.get('grain_risk') in ('high', 'medium'):
                log(f"    → {gc.get('fragment_name')}: {gc.get('grain_risk')} risk, remedy={gc.get('remedy')}")
    else:
        log(f"  [GRAIN CHECK] ✓ No grain inflation risks detected")

    # ===== Step 2b: Filter Grounding + lookup_val Planning =====
    step2b = call_llm(
        STEP2B_SYSTEM,
        STEP2B_PROMPT.format(
            SCHEMA_TEXT=schema,
            MAPPING_HINTS=mapping_hints,
            STEP1A_JSON=json.dumps(step1a, ensure_ascii=False),
            STEP1B_JSON=json.dumps(step1b, ensure_ascii=False),
            SEMANTIC_HINTS=semantic_hints
        ),
        step_name="step2b",
        model="gpt-4o",
    )
    steps['step2b'] = step2b

    # Execute lookup_val and PROGRAMMATICALLY apply results
    log(f"\n  [LOOKUP_VAL PHASE]")
    lookup_results = run_lookup_val(step2b, DB_CONFIG, db_id=db_id)
    if lookup_results:
        steps['lookup_results'] = lookup_results
        log(f"  [LOOKUP_VAL] {len(lookup_results)} filters resolved")

        # Programmatic injection (replaces unreliable LLM update)
        step2b_updated, changes = apply_lookup_results(step2b, lookup_results)
        if changes:
            log(f"  [PROGRAMMATIC UPDATE] Applied {len(changes)} changes:")
            for c in changes:
                log(f"    → {c}")
            steps['step2b_updated'] = step2b_updated
            step2b = step2b_updated  # Use updated version downstream
        else:
            log(f"  [PROGRAMMATIC UPDATE] No changes applied (lookup had no actionable results)")
    else:
        log(f"  [LOOKUP_VAL] No lookup plans found, skipping.")

    # ===== GUARD: SELECT 컬럼과 동일한 컬럼을 WHERE로 쓰는 필터 제거 =====
    select_columns = set()
    for mc in step2a.get('mapping_candidates', []):
        for c in mc.get('candidates', []):
            expr = c.get('expr', '').upper()
            # 함수가 아닌 직접 컬럼 참조만 (e.g., "BUILDINGS.BUILDING_NAME")
            if '(' not in expr and '.' in expr:
                select_columns.add(expr)

    if select_columns and step2b.get('resolved_filters'):
        original_count = len(step2b['resolved_filters'])
        kept_filters = []
        for rf in step2b['resolved_filters']:
            rec = rf.get('final_recommendation', {})
            if not isinstance(rec, dict):
                rec = {}
            tbl = rec.get('table', '')
            col = rec.get('column', '')
            col_ref = f"{tbl}.{col}".upper() if tbl and col else ''
            if col_ref and col_ref in select_columns:
                log(f"  [SELECT-FILTER GUARD] Removed filter '{rf.get('filter_name','')}': {col_ref} is a SELECT column, not a WHERE filter")
            else:
                kept_filters.append(rf)
        if len(kept_filters) < original_count:
            step2b['resolved_filters'] = kept_filters
            log(f"  [SELECT-FILTER GUARD] {original_count - len(kept_filters)} filter(s) removed")

    # ===== Step 2c: GROUP BY vs WINDOW Decision (now includes mapping_hints) =====
    step2c = call_llm(
        STEP2C_SYSTEM,
        STEP2C_PROMPT.format(
            MAPPING_HINTS=mapping_hints,
            STEP1A_JSON=json.dumps(step1a, ensure_ascii=False),
            SELECT_MAPPING_JSON=json.dumps(step2a, ensure_ascii=False),
            FILTER_GROUNDING_JSON=json.dumps(step2b, ensure_ascii=False),
            STEP1B_JSON=json.dumps(step1b, ensure_ascii=False),
            GRAIN_CHECKS_JSON=json.dumps(step2a_g, ensure_ascii=False)
        ),
        step_name="step2c",
        model="gpt-4o",
    )
    steps['step2c'] = step2c

    # ===== Step 3: Final Assembly =====
    assembled_spec = call_llm(
        STEP3_SYSTEM,
        STEP3_PROMPT.format(
            QUESTION=question,
            STEP1A_JSON=json.dumps(step1a, ensure_ascii=False),
            STEP2A_JSON=json.dumps(step2a, ensure_ascii=False),
            STEP2B_JSON=json.dumps(step2b, ensure_ascii=False),
            STEP2C_JSON=json.dumps(step2c, ensure_ascii=False),
            GRAIN_CHECKS_JSON=json.dumps(step2a_g, ensure_ascii=False),
            SEMANTIC_HINTS=semantic_hints if semantic_hints else "(none)",
            MAPPING_HINTS=mapping_hints,
            JOIN_INFO=join_info
        ),
        step_name="step3",
        model="gpt-4o",
        json_mode=False,
        max_tokens=3000
    )
    steps['step3_assembled'] = assembled_spec

    # ===== Step 4: Final SQL Rendering =====
    sql_response = call_llm(
        STEP4_SYSTEM,
        STEP4_PROMPT.format(
            ASSEMBLED_SPEC=assembled_spec,
            SCHEMA_TEXT=schema
        ),
        step_name="step4",
        model="gpt-4o",
        json_mode=False,
        max_tokens=3000
    )

    # Extract SQL from markdown code blocks if present
    predicted_sql = sql_response.strip()
    if predicted_sql.startswith("```sql"):
        predicted_sql = predicted_sql[6:]
    elif predicted_sql.startswith("```"):
        predicted_sql = predicted_sql[3:]
    if predicted_sql.endswith("```"):
        predicted_sql = predicted_sql[:-3]
    predicted_sql = predicted_sql.strip()

    # Post-process: clean up any placeholder variables or invalid patterns
    predicted_sql = clean_sql(predicted_sql)

    log(f"\n  [PREDICTED SQL (before refine)] {predicted_sql}")

    # ===== Step 5: Refine (execute SQL → analyze error → LLM fix → retry up to 2 times) =====
    log(f"\n  [STEP 5: REFINE]")
    refined_sql, refine_log = refine_sql(predicted_sql, item, max_retries=2)
    steps['refine_log'] = refine_log

    if refined_sql != predicted_sql:
        log(f"\n  [REFINED] SQL was modified by refine step")
        steps['original_predicted_sql'] = predicted_sql
        predicted_sql = refined_sql

    # ===== Structural Validation (Grain Safety) =====
    if has_grain_risks:
        grain_validation = validate_grain_safety(predicted_sql, step2a_g, item)
        steps['grain_validation'] = grain_validation
        if not grain_validation.get('validated', True):
            log(f"\n  [GRAIN VALIDATION] ⚠️ Structural risk: {grain_validation.get('flagged_count', 0)} aggregate(s) may have inflation")
            for chk in grain_validation.get('checks', []):
                log(f"    → {chk['fragment']}: {chk['grain_risk']} risk — remedy: {chk['remedy']}")
        else:
            log(f"\n  [GRAIN VALIDATION] ✓ No structural issues detected")

    log(f"\n  [FINAL SQL] {predicted_sql}")
    log(f"\n  [GOLD SQL] {gold_query}")

    return {
        "db_id": db_id,
        "question": question,
        "predicted_sql": predicted_sql,
        "gold_query": gold_query,
        "original_index": original_index,
        "steps": steps
    }


# ============================================================
# PIPELINE EXECUTION
# ============================================================
log_buffer.clear()
log(f"Pipeline Experiment (7-Step Decomposed v5 + Refine) | {LOG_TIMESTAMP} | Items: {len(dataset)}")
log(f"USE_SEMANTIC_LAYER: {USE_SEMANTIC_LAYER}")

results = []
for item in tqdm(dataset, desc="Processing (7-step v5 + refine)"):
    result = process_item(item, verbose=False)
    results.append(result)

# Save log
flush_log()

print(f"\nProcessed {len(results)} items")

Processing (7-step v5 + refine): 100%|██████████| 20/20 [11:15<00:00, 33.77s/it]

Log saved: /Users/kyong/Desktop/text-to-sql-analysis/scripts/pipeline/pipeline_log_20260225_093115.txt

Processed 20 items


## 5. Save Predictions (for evaluation.py)

In [19]:
# Save predictions.json (evaluation.py 호환 형식)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_output_dir = os.path.join(OUTPUT_DIR, f"exp_{timestamp}")
os.makedirs(exp_output_dir, exist_ok=True)

predictions_for_eval = []
for r in results:
    predictions_for_eval.append({
        "db_id": r["db_id"],
        "question": r["question"],
        "predicted_sql": r["predicted_sql"],
        "original_index": r.get("original_index")
    })

predictions_path = os.path.join(exp_output_dir, "predictions.json")
with open(predictions_path, 'w', encoding='utf-8') as f:
    json.dump(predictions_for_eval, f, indent=2, ensure_ascii=False)

print(f"Predictions saved to: {predictions_path}")
print(f"\nTo evaluate, run:")
print(f"  python evaluate.py --prediction_path {predictions_path} --config configs/beaver_dw_openai.yaml")

Predictions saved to: /Users/kyong/Desktop/text-to-sql-analysis/outputs/pipeline_experiment/exp_20260225_094231/predictions.json

To evaluate, run:
  python evaluate.py --prediction_path /Users/kyong/Desktop/text-to-sql-analysis/outputs/pipeline_experiment/exp_20260225_094231/predictions.json --config configs/beaver_dw_openai.yaml


## 6. Quick Evaluation (Execution Accuracy)

In [20]:
import mysql.connector

def execute_sql(db_id: str, sql: str, limit: int = 50) -> dict:
    """Execute SQL and return results."""
    try:
        conn = mysql.connector.connect(
            host=DB_CONFIG['host'],
            port=DB_CONFIG['port'],
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password'],
            database=db_id,
            consume_results=True
        )
        cursor = conn.cursor(dictionary=True, buffered=True)
        cursor.execute(sql)
        rows = cursor.fetchmany(limit)
        cursor.close()
        conn.close()
        return {"success": True, "rows": rows, "row_count": len(rows)}
    except Exception as e:
        return {"success": False, "error": str(e)[:300]}

def compare_results(db_id: str, gold_sql: str, pred_sql: str) -> dict:
    """Compare execution results."""
    gold_result = execute_sql(db_id, gold_sql)
    pred_result = execute_sql(db_id, pred_sql)
    
    if not gold_result.get('success') or not pred_result.get('success'):
        return {
            'gold_success': gold_result.get('success', False),
            'pred_success': pred_result.get('success', False),
            'pred_error': pred_result.get('error', ''),
            'exact_match': False
        }
    
    gold_rows = gold_result.get('rows', [])
    pred_rows = pred_result.get('rows', [])
    
    gold_set = set(tuple(sorted(str(v) for v in row.values())) for row in gold_rows)
    pred_set = set(tuple(sorted(str(v) for v in row.values())) for row in pred_rows)
    
    return {
        'gold_success': True,
        'pred_success': True,
        'exact_match': gold_set == pred_set,
        'gold_rows': len(gold_rows),
        'pred_rows': len(pred_rows)
    }

In [21]:
# Run quick evaluation
log(f"\n{'#'*70}")
log(f"# EVALUATION")
log(f"{'#'*70}")

eval_results = []

for r in tqdm(results, desc="Evaluating"):
    if r['predicted_sql'].startswith('Error:'):
        comparison = {'exact_match': False, 'pred_success': False, 'pred_error': r['predicted_sql']}
    else:
        comparison = compare_results(r['db_id'], r['gold_query'], r['predicted_sql'])
    
    eval_results.append({
        'index': r['original_index'],
        'question': r['question'],
        'exact_match': comparison.get('exact_match', False),
        'pred_success': comparison.get('pred_success', False),
        'gold_rows': comparison.get('gold_rows', 0),
        'pred_rows': comparison.get('pred_rows', 0),
        'error': comparison.get('pred_error', '')
    })

# Summary
correct = sum(1 for e in eval_results if e['exact_match'])
exec_success = sum(1 for e in eval_results if e['pred_success'])
total = len(eval_results)

summary_text = f"""
{'='*60}
ACCURACY: {correct}/{total} ({correct/total*100:.1f}%)
Execution Success: {exec_success}/{total}
{'='*60}"""

print(summary_text)
log(summary_text)

# Per-item resultsㅋ
for e in eval_results:
    if e['exact_match']:
        status = "CORRECT"
    elif e['pred_success']:
        status = "WRONG"
    else:
        status = "ERROR"
    line = f"  [{e['index']:3d}] {status:8s} - {e['question']}..."
    print(line)
    log(line)

# Save updated log with evaluation
flush_log()

Evaluating: 100%|██████████| 20/20 [00:17<00:00,  1.13it/s]


ACCURACY: 5/20 (25.0%)
Execution Success: 18/20
  [ 20] ERROR    - What is the building component, name of the building, square footage for all rooms, total number of floors, total number of rooms, total number of facility organizations, total number of supervisors, and total number of supervisees for each building component?...
  [ 21] ERROR    - What is the DLC key, name of the DLC, total number of floors, total square footage, total number of facility organizations, total number of supervisors, total number of supervisees, and total building heights for each DLC?...
  [ 22] CORRECT  - What is the department name, total number of types of TIP subjects, total number of enrolled students, the minimum and maximum rental new price for each department?...
  [ 23] WRONG    - What are the details of courses offered in the current academic term, including the academic year, term code, hgn code, the total number of types of courses, the average number of units, the department name, the name 